In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

In [2]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/facebook-recruiting-iv-human-or-bot/train.csv.zip
/kaggle/input/facebook-recruiting-iv-human-or-bot/sampleSubmission.csv
/kaggle/input/facebook-recruiting-iv-human-or-bot/bids.csv.zip
/kaggle/input/facebook-recruiting-iv-human-or-bot/test.csv.zip


In [3]:
bidder_train = pd.read_csv("/kaggle/input/facebook-recruiting-iv-human-or-bot/train.csv.zip")
bidder_test = pd.read_csv("/kaggle/input/facebook-recruiting-iv-human-or-bot/test.csv.zip")
bids = pd.read_csv("/kaggle/input/facebook-recruiting-iv-human-or-bot/bids.csv.zip")

sampleSubmission = pd.read_csv("/kaggle/input/facebook-recruiting-iv-human-or-bot/sampleSubmission.csv")

In [4]:
print(f"bidder_train.shape: {bidder_train.shape}")
print(f"bidder_train.columns: {list(bidder_train.columns)}")
bidder_train.sample(2)

bidder_train.shape: (2013, 4)
bidder_train.columns: ['bidder_id', 'payment_account', 'address', 'outcome']


,bidder_id,payment_account,address,outcome
1738,c615c31c87260c0d85727deff6da6667lgg33,a3d2de7675556553a5f08e4c88d2c22828v1t,a3d2de7675556553a5f08e4c88d2c228liatf,0.0
778,ae15accae2164aae084aef7e3524c1fflq962,a3d2de7675556553a5f08e4c88d2c228vu63m,a3d2de7675556553a5f08e4c88d2c228fvznj,0.0


In [5]:
print(f"bidder_test.shape: {bidder_test.shape}")
print(f"bidder_test.columns: {list(bidder_test.columns)}")
bidder_test.sample(2)

bidder_test.shape: (4700, 3)
bidder_test.columns: ['bidder_id', 'payment_account', 'address']


,bidder_id,payment_account,address
300,55a699bd3a83fe99db348c4302ee754ep392w,36929b6342fb38e84567d61b38fbada90728u,a3d2de7675556553a5f08e4c88d2c228xcc2k
1795,b24bd373016937b0294b3490815ca8d6tzksi,bdea3ff0541fe379bd07a7b0f2694df45996q,b6c7bfd9c22b3ba3e465bedca137f9e4dqk3f


In [6]:
print(f"bids.shape: {bids.shape}")
print(f"bids.columns: {list(bids.columns)}")
bids.sample(2)

bids.shape: (7656334, 9)
bids.columns: ['bid_id', 'bidder_id', 'auction', 'merchandise', 'device', 'time', 'country', 'ip', 'url']


,bid_id,bidder_id,auction,merchandise,device,time,country,ip,url
941907,941907,6c8ffec692e88d81e1b18d47818abb04v6llr,veq9j,home goods,phone57,9764321000000000,za,249.126.74.36,o9t0pgzh9piziot
7471019,7471019,9328bc6389d718c41e33c4cefe7b9c15k1tau,2pxeq,home goods,phone2075,9708307263157894,us,76.2.175.169,vasstdc27m7nks3


In [7]:
print(f"sampleSubmission.shape: {sampleSubmission.shape}")
print(f"sampleSubmission.columns: {list(sampleSubmission.columns)}")
sampleSubmission.sample(2)

sampleSubmission.shape: (4700, 2)
sampleSubmission.columns: ['bidder_id', 'prediction']


,bidder_id,prediction
2941,8ae8d02450efade0ce821b34c0461e21gs1to,0.0
1316,ea4249ac7ccb24fcb8f9f1a2bf672f87qadms,0.0


In [8]:
def build_bidder_features(bids: pd.DataFrame) -> pd.DataFrame:
    g = bids.groupby("bidder_id")

    # time features
    tmin = g["time"].min().rename("time_min")
    tmax = g["time"].max().rename("time_max")
    n    = g.size().rename("n_bids")
    span = (tmax - tmin).rename("time_span")
    avg_gap = (span / np.maximum(n - 1, 1)).rename("avg_gap")

    # diversity
    n_auc   = g["auction"].nunique().rename("n_auctions")
    n_dev   = g["device"].nunique().rename("n_devices")
    n_ctry  = g["country"].nunique().rename("n_countries")
    n_ip    = g["ip"].nunique().rename("n_ips")
    n_url   = g["url"].nunique().rename("n_urls")
    n_merch = g["merchandise"].nunique().rename("n_merchandise")

    # intensity
    bids_per_auc = (n / np.maximum(n_auc, 1)).rename("bids_per_auction")

    # popularity
    auc_pop = bids["auction"].value_counts()
    dev_pop = bids["device"].value_counts()
    mean_auc_pop = g["auction"].apply(lambda s: auc_pop.loc[s].mean()).rename("mean_auc_pop")
    mean_dev_pop = g["device"].apply(lambda s: dev_pop.loc[s].mean()).rename("mean_dev_pop")

    bids_sorted = bids.sort_values(["bidder_id", "time"]).copy()
    bids_sorted["gap"] = bids_sorted.groupby("bidder_id")["time"].diff()

    gap_median = bids_sorted.groupby("bidder_id")["gap"] \
        .median().fillna(0).rename("gap_median")

    switch_rate_device = bids_sorted.groupby("bidder_id")["device"] \
        .apply(lambda s: (s.iloc[1:].values != s.iloc[:-1].values).mean() if len(s) > 1 else 0.0) \
        .rename("switch_rate_device")

    feats = pd.concat(
        [
            n, span, avg_gap,
            n_auc, n_dev, n_ctry, n_ip, n_url, n_merch,
            bids_per_auc,
            mean_auc_pop, mean_dev_pop,
            gap_median,
            switch_rate_device
        ],
        axis=1
    ).reset_index()

    return feats

bidder_feats = build_bidder_features(bids)

In [9]:
round(100 * (bidder_feats.isnull().sum() / len(bidder_feats)), 3)

bidder_id             0.0
n_bids                0.0
time_span             0.0
avg_gap               0.0
n_auctions            0.0
n_devices             0.0
n_countries           0.0
n_ips                 0.0
n_urls                0.0
n_merchandise         0.0
bids_per_auction      0.0
mean_auc_pop          0.0
mean_dev_pop          0.0
gap_median            0.0
switch_rate_device    0.0
dtype: float64

In [10]:
# 1) How many bidders have no bids (no match)?
m = bidder_train.merge(bidder_feats, on="bidder_id", how="left", indicator=True)
print(m["_merge"].value_counts())  # look for 'left_only'

n = bidder_test.merge(bidder_feats, on="bidder_id", how="left", indicator=True)
print(n["_merge"].value_counts())  # look for 'left_only'

_merge
both          1984
left_only       29
right_only       0
Name: count, dtype: int64
_merge
both          4630
left_only       70
right_only       0
Name: count, dtype: int64


In [11]:
# 2) Ensure exact key match
bidder_train["bidder_id"] = bidder_train["bidder_id"].astype(str).str.strip()
bidder_feats["bidder_id"]  = bidder_feats["bidder_id"].astype(str).str.strip()

In [12]:
# 3) After merging, fill feature NaNs with 0 (or sensible defaults)
train_df = bidder_train.merge(bidder_feats, on="bidder_id", how="left").fillna(0)
test_df  = bidder_test.merge(bidder_feats,  on="bidder_id", how="left").fillna(0)

In [13]:
train_df.head()

,bidder_id,payment_account,address,outcome,n_bids,time_span,avg_gap,n_auctions,n_devices,n_countries,n_ips,n_urls,n_merchandise,bids_per_auction,mean_auc_pop,mean_dev_pop,gap_median,switch_rate_device
0,91a3c57b13234af24875c56fb7e2b2f4rb56a,a3d2de7675556553a5f08e4c88d2c228754av,a3d2de7675556553a5f08e4c88d2c228vt0u4,0.0,24.0,1.313558e+13,5.711121e+11,18.0,14.0,6.0,20.0,1.0,1.0,1.333333,4321.333333,212995.583333,3.458421e+11,0.913043
1,624f258b49e77713fc34034560f93fb3hu3jo,a3d2de7675556553a5f08e4c88d2c228v1sga,ae87054e5a97a8f840a3991d12611fdcrfbq3,0.0,3.0,6.467158e+12,3.233579e+12,1.0,2.0,1.0,3.0,2.0,1.0,3.000000,208926.000000,6751.333333,3.233579e+12,0.500000
2,1c5f4fc669099bfbfac515cd26997bd12ruaj,a3d2de7675556553a5f08e4c88d2c2280cybl,92520288b50f03907041887884ba49c0cl0pd,0.0,4.0,7.137000e+12,2.379000e+12,4.0,2.0,1.0,4.0,2.0,1.0,1.000000,920.750000,532243.500000,2.532053e+12,0.333333
3,4bee9aba2abda51bf43d639013d6efe12iycd,51d80e233f7b6a7dfdee484a3c120f3b2ita8,4cb9717c8ad7e88a9a284989dd79b98dbevyi,0.0,1.0,0.000000e+00,0.000000e+00,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,1715.000000,11220.000000,0.000000e+00,0.000000
4,4ab12bc61c82ddd9c2d65e60555808acqgos1,a3d2de7675556553a5f08e4c88d2c22857ddh,2a96c3ce94b3be921e0296097b88b56a7x1ji,0.0,155.0,1.197795e+13,7.777888e+10,23.0,53.0,2.0,123.0,91.0,1.0,6.739130,11214.374194,151573.877419,1.368421e+10,0.837662


In [14]:
test_df.head()

,bidder_id,payment_account,address,n_bids,time_span,avg_gap,n_auctions,n_devices,n_countries,n_ips,n_urls,n_merchandise,bids_per_auction,mean_auc_pop,mean_dev_pop,gap_median,switch_rate_device
0,49bb5a3c944b8fc337981cc7a9ccae41u31d7,a3d2de7675556553a5f08e4c88d2c228htx90,5d9fa1b71f992e7c7a106ce4b07a0a754le7c,4.0,7.022368e+13,2.340789e+13,3.0,2.0,3.0,4.0,3.0,1.0,1.333333,273237.000000,119993.750000,5.781053e+12,0.333333
1,a921612b85a1494456e74c09393ccb65ylp4y,a3d2de7675556553a5f08e4c88d2c228rs17i,a3d2de7675556553a5f08e4c88d2c228klidn,3.0,7.600205e+13,3.800103e+13,2.0,3.0,2.0,2.0,1.0,1.0,1.500000,624.000000,14692.333333,3.800103e+13,1.000000
2,6b601e72a4d264dab9ace9d7b229b47479v6i,925381cce086b8cc9594eee1c77edf665zjpl,a3d2de7675556553a5f08e4c88d2c228aght0,17.0,2.910526e+11,1.819079e+10,14.0,4.0,3.0,4.0,2.0,1.0,1.214286,7565.352941,43711.941176,2.315789e+09,0.375000
3,eaf0ed0afc9689779417274b4791726cn5udi,a3d2de7675556553a5f08e4c88d2c228nclv5,b5714de1fd69d4a0d2e39d59e53fe9e15vwat,148.0,7.652163e+13,5.205553e+11,90.0,81.0,14.0,129.0,80.0,1.0,1.644444,2601.716216,136240.770270,1.046842e+11,0.911565
4,cdecd8d02ed8c6037e38042c7745f688mx5sf,a3d2de7675556553a5f08e4c88d2c228dtdkd,c3b363a3c3b838d58c85acf0fc9964cb4pnfa,23.0,6.574789e+12,2.988541e+11,20.0,17.0,2.0,17.0,1.0,1.0,1.150000,756.565217,33868.913043,8.131579e+09,1.000000


In [15]:
drop_cols = ["bidder_id", "payment_account", "address", "outcome"]
X = train_df.drop(columns=drop_cols, errors="ignore")
y = train_df["outcome"].astype(int)

In [16]:
X.head()

,n_bids,time_span,avg_gap,n_auctions,n_devices,n_countries,n_ips,n_urls,n_merchandise,bids_per_auction,mean_auc_pop,mean_dev_pop,gap_median,switch_rate_device
0,24.0,1.313558e+13,5.711121e+11,18.0,14.0,6.0,20.0,1.0,1.0,1.333333,4321.333333,212995.583333,3.458421e+11,0.913043
1,3.0,6.467158e+12,3.233579e+12,1.0,2.0,1.0,3.0,2.0,1.0,3.000000,208926.000000,6751.333333,3.233579e+12,0.500000
2,4.0,7.137000e+12,2.379000e+12,4.0,2.0,1.0,4.0,2.0,1.0,1.000000,920.750000,532243.500000,2.532053e+12,0.333333
3,1.0,0.000000e+00,0.000000e+00,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,1715.000000,11220.000000,0.000000e+00,0.000000
4,155.0,1.197795e+13,7.777888e+10,23.0,53.0,2.0,123.0,91.0,1.0,6.739130,11214.374194,151573.877419,1.368421e+10,0.837662


In [17]:
# 1) stratified holdout
X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2) RF + randomized search (AUC)
rf = RandomForestClassifier(n_jobs=-1, random_state=42)

param_dist = {
    "n_estimators": randint(300, 2000),
    "max_depth": randint(3, 30),
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
    "max_features": ["sqrt", "log2", 0.5, 0.7, 0.9],
    "bootstrap": [True],
    "class_weight": [None, "balanced"]  # helpful for imbalance
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=40,                
    scoring="roc_auc",
    n_jobs=-1,
    cv=cv,
    verbose=0,
    random_state=42,
    refit=True                
)

search.fit(X_tr, y_tr)
print("Best CV AUC (inner CV):", search.best_score_)
print("Best params:", search.best_params_)

# 3) evaluate best model on the holdout
best_rf = search.best_estimator_
va_pred = best_rf.predict_proba(X_va)[:, 1]
print("Holdout AUC:", roc_auc_score(y_va, va_pred))

# final fit on all data for test predictions
final_rf = RandomForestClassifier(**{**search.best_params_, "n_jobs": -1, "random_state": 42})
final_rf.fit(X, y)

cols = X.columns
test_pred = final_rf.predict_proba(test_df[cols])[:, 1]

Best CV AUC (inner CV): 0.9291291653273331
Best params: {'bootstrap': True, 'class_weight': None, 'max_depth': 11, 'max_features': 'log2', 'min_samples_leaf': 3, 'min_samples_split': 8, 'n_estimators': 1564}
Holdout AUC: 0.9415357766143106


In [18]:
sub = pd.DataFrame({"bidder_id": test_df["bidder_id"], "prediction": test_pred})
sub.to_csv("submission.csv", index=False)
print("Saved submission.csv")

Saved submission.csv
